Week 12 - Build a basic function-calling agent that retrieves and processes weather information, demonstrating the practical implementation of agent capabilities

In [5]:
import os

os.environ['WEATHER_API_KEY']='WEATHER_API_KEY'

In [6]:
#Part 1: Setting Up Weather API Integration
import requests

def get_weather(location):
    try:
        WEATHER_API_KEY = os.getenv('WEATHER_API_KEY')
        url = f"http://api.weatherapi.com/v1/current.json"
        params = {
            "key": WEATHER_API_KEY,
            "q": location,
            "aqi": "no"
        }
        response = requests.get(url, params=params)
        response.raise_for_status()  # Raise exception for bad status codes
        return response.json()
    except requests.exceptions.RequestException as e:
        raise Exception(f"Weather API error: {str(e)}")

def test(location):
    try:
      weather_data = get_weather(location)
      print(f"Weather in {weather_data['location']['name']}:")
      print(f"Temperature: {weather_data['current']['temp_c']}°C")
      print(f"Condition: {weather_data['current']['condition']['text']}")
    except Exception as e:
      print(f"Error: {e}")

In [7]:
#Test Part 1:

test("London")

Weather in London:
Temperature: 10.0°C
Condition: Moderate rain


In [8]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (1,634 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current u

In [9]:
!nohup ollama serve &

nohup: appending output to 'nohup.out'


In [10]:
!ollama -v
!ollama pull llama3.2
!ollama list

ollama version is 0.20.2

NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    


In [15]:
# Adding OpenAI Function Definition
import openai

# Define the function that OpenAI can call
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "City name or location"},
                },
                "required": ["location"],
            },
        },
    }
]

functions = [
    {
        "name": "get_weather",
        "description": "Get current weather for a location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name or location"
                }
            },
            "required": ["location"]
        }
    }
]

def test2():
    try:
        from openai import OpenAI

        client = OpenAI(
              base_url='http://localhost:11434/v1/',
              api_key='ollama', # Required but ignored by Ollama
        )
        response = client.chat.completions.create(
            model="llama3.2",
            messages=[{"role": "user", "content": "What's the weather like in Tokyo?"}],
            #functions=functions,
            #function_call="auto" // Function call not working. OpenAI giving its own response.
            tools=tools,
            #tool_choice="auto",
            verbosity='medium'
        )
        print("OpenAI Response:")
        #print("Function Call:")
        #print(response.choices[0].message.function_call)
        print("Tool Call:")
        print(response.choices[0].message.tool_calls[0].function.name)
        print(response.choices[0].message.tool_calls[0].function.arguments)
    except Exception as e:
        print(f"Error: {e}")

#Test Part 2
test2()

OpenAI Response:
Tool Call:
get_weather
{"location":"Tokyo"}


In [26]:
#Part 3: Implementing the Weather Agent
from openai import OpenAI

def weather_agent(user_query):
    try:
        # Get function call from OpenAI
        client = OpenAI(
              base_url='http://localhost:11434/v1/',
              api_key='ollama', # Required but ignored by Ollama
        )
        response = client.chat.completions.create(
            model="llama3.2",
            messages=[{"role": "user", "content": user_query}],
            #functions=functions,
            #function_call="auto",
            tools=tools,
            #tool_choice="auto",
            verbosity='medium'
        )
        print(response)
        # Extract function call
        #function_call = response.choices[0].message.function_call
        function_call = response.choices[0].message.tool_calls[0].function
        print(function_call)
        # Execute function
        if function_call.name == "get_weather":
            # Parse the arguments from JSON string
            import json
            arguments = json.loads(function_call.arguments)
            weather_data = get_weather(arguments["location"])

            # Format response
            result = f"Current weather in {weather_data['location']['name']}: {weather_data['current']['temp_c']}°C, {weather_data['current']['condition']['text']}"
            return result
    except Exception as e:
        return f"Error: {str(e)}"

In [27]:
#Final Test:

# Test the complete weather agent
test_queries = [
    "What's the weather like in Singapore?",
    "Is it raining in London right now?",
    "Tell me the temperature in New York"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    print(f"Response: {weather_agent(query)}")


Query: What's the weather like in Singapore?
ChatCompletion(id='chatcmpl-425', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_gbhimu73', function=Function(arguments='{"location":"Singapore"}', name='get_weather'), type='function', index=0)]))], created=1775382505, model='llama3.2', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=17, prompt_tokens=159, total_tokens=176, completion_tokens_details=None, prompt_tokens_details=None))
Function(arguments='{"location":"Singapore"}', name='get_weather')
Response: Current weather in Singapore: 27.1°C, Partly cloudy

Query: Is it raining in London right now?
ChatCompletion(id='chatcmpl-963', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatComplet